# Retrieval baselines

Compare text and entity-context retrieval, then audit split overlap.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.data import write_json

pd.set_option("display.max_columns", 20)

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from src.text_extraction import COMPILED
from src.retrieval import mask_identifiers, normalize_template, build_graph_features, fit_representations, rank_candidates
from src.evaluation import evaluate_rankings

records = pd.read_csv(ROOT / "results/text/record_audit.csv", keep_default_na=False)
entities = pd.read_csv(ROOT / "results/text/entities.csv", keep_default_na=False)
graph_edges = pd.read_csv(ROOT / "results/kg/edges.csv", keep_default_na=False)
out = ROOT / "results/retrieval/diagnostic"
out.mkdir(parents=True, exist_ok=True)

<local environment>:22: DtypeWarning: Columns (3: row_id, 9: span_start, 10: span_end) have mixed types. Specify dtype option on import or set low_memory=False.
  graph_edges = pd.read_csv(ROOT / "results/kg/edges.csv", keep_default_na=False)


## Baseline task

We first use single extracted failure modes as a diagnostic retrieval target. They come from the descriptions themselves: even after removing failure terms, this evaluates consistency with a lexical rule rather than independently observed failures.

In [2]:
fields = ["equipment_id", "order_type", "maintenance_type", "problem_text", "action_text", "combined_text"]
conflicts = records.groupby("work_order")[fields].nunique().gt(1).any(axis=1)
base = records.loc[~records["work_order"].isin(conflicts.index[conflicts])].drop_duplicates("work_order").copy()
failure_labels = entities.loc[entities["entity_type"].eq("failure_mode")].groupby("row_id")["entity_label"].agg(lambda x: sorted(set(x)))
single = failure_labels[failure_labels.map(len).eq(1)].map(lambda x: x[0]).rename("failure_mode")
cohort = base.merge(single, left_on="row_id", right_index=True, validate="one_to_one")
cohort = cohort.loc[cohort["equipment_id"].ne("")].copy()
counts = cohort["failure_mode"].value_counts()
cohort = cohort.loc[cohort["failure_mode"].isin(counts[counts >= 50].index)].sort_values("row_id").reset_index(drop=True)
assert not cohort.empty
cohort["template"] = cohort["combined_text"].map(normalize_template)
masked = cohort["combined_text"].copy()
for pattern in COMPILED["failure_mode"].values():
    masked = masked.map(lambda text: pattern.sub(" failure ", text))
cohort["masked_text"] = masked.map(mask_identifiers)
display(cohort["failure_mode"].value_counts())

failure_mode
wear               3117
leak               3079
trip               2915
blockage           2021
bearing_failure     634
pressure_loss       487
Name: count, dtype: int64

## Random-split baseline

In [3]:
def benchmark(train_idx, test_idx, split_name):
    candidates = cohort.iloc[train_idx].sort_values("row_id").reset_index(drop=True)
    queries = cohort.iloc[test_idx].copy()
    if len(queries) > config.MAX_QUERIES:
        queries = queries.sample(config.MAX_QUERIES, random_state=config.SEED)
    queries = queries.sort_values("row_id").reset_index(drop=True)
    candidate_features = build_graph_features(candidates, graph_edges, include_failure=False)
    query_features = build_graph_features(queries, graph_edges, include_failure=False)
    ct, qt, cg, qg = fit_representations(candidates["masked_text"], queries["masked_text"], candidate_features, query_features)
    rankings, hybrid, pool = rank_candidates([ct, cg], [qt, qg], candidates, queries)
    frequencies = candidates["failure_mode"].value_counts()
    tie_order = np.random.default_rng(config.SEED).permutation(len(candidates))
    prior_scores = candidates["failure_mode"].map(frequencies).to_numpy()
    prior = tie_order[np.argsort(-prior_scores[tie_order], kind="stable")][:50]
    methods = {"Prior": [prior] * len(queries), "TF-IDF": rankings[0], "Graph context": rankings[1], "Hybrid RRF": hybrid}
    per_query = pd.concat([evaluate_rankings(ranking, queries, candidates, "failure_mode", method)
                           for method, ranking in methods.items()], ignore_index=True)
    per_query["split"] = split_name
    audit = {
        "split": split_name, "candidate_records": len(candidates), "query_records": len(queries),
        "full_test_records": len(test_idx), "train_classes": candidates["failure_mode"].nunique(),
        "test_classes": queries["failure_mode"].nunique(),
        "unsupported_queries": int((~queries["failure_mode"].isin(candidates["failure_mode"])).sum()),
        "exact_text_overlap": len(set(candidates["combined_text"]) & set(queries["combined_text"])),
        "template_overlap": len(set(candidates["template"]) & set(queries["template"])),
        "equipment_overlap": len(set(candidates["equipment_id"]) & set(queries["equipment_id"])),
        "row_overlap": len(set(candidates["row_id"]) & set(queries["row_id"])),
        "representation_fit": "candidate records only", "failure_nodes_in_features": False
    }
    membership = pd.concat([candidates[["row_id"]].assign(role="candidate"),
                            queries[["row_id"]].assign(role="query")], ignore_index=True).assign(split=split_name)
    return per_query, audit, membership

train_idx, test_idx = train_test_split(np.arange(len(cohort)), test_size=config.TEST_SIZE,
                                      random_state=config.SEED, stratify=cohort["failure_mode"])
random_results, random_audit, random_membership = benchmark(train_idx, test_idx, "Random")
metric_cols = [f"hit@{k}" for k in config.TOP_K] + ["mrr@10", "mrr@50"]
display(random_results.groupby("method")[metric_cols].mean())

,hit@1,hit@5,hit@10,hit@20,hit@50,mrr@10,mrr@50
method,,,,,,,
Graph context,0.988000,0.996667,0.998000,0.998667,0.998667,0.991689,0.991737
Hybrid RRF,0.998000,1.000000,1.000000,1.000000,1.000000,0.998667,0.998667
Prior,0.258667,0.258667,0.258667,0.258667,0.258667,0.258667,0.258667
TF-IDF,0.999333,1.000000,1.000000,1.000000,1.000000,0.999667,0.999667


## Overlap audit and stricter splits

The random split is now audited for repeated text, normalized templates and shared equipment. Group-disjoint splits test how much this diagnostic depends on those overlaps; each uses the fixed seed once, without searching for a favorable class distribution.

In [4]:
display(pd.DataFrame([random_audit]))
all_results, all_audits, memberships = [random_results], [random_audit], [random_membership]
for name, group in [("Template-disjoint", "template"), ("Equipment-disjoint", "equipment_id")]:
    splitter = GroupShuffleSplit(n_splits=1, test_size=config.TEST_SIZE, random_state=config.SEED)
    train_idx, test_idx = next(splitter.split(cohort, groups=cohort[group]))
    result, audit, membership = benchmark(train_idx, test_idx, name)
    assert audit["row_overlap"] == 0
    assert audit["template_overlap" if group == "template" else "equipment_overlap"] == 0
    all_results.append(result)
    all_audits.append(audit)
    memberships.append(membership)
results = pd.concat(all_results, ignore_index=True)
audits = pd.DataFrame(all_audits)
metrics = results.groupby(["split", "method"])[metric_cols].mean().reset_index()
results.to_csv(out / "per_query.csv", index=False)
metrics.to_csv(out / "metrics.csv", index=False)
audits.to_csv(out / "split_audit.csv", index=False)
pd.concat(memberships, ignore_index=True).to_csv(out / "split_membership.csv", index=False)
display(metrics.round(4))
display(audits)

,split,candidate_records,query_records,full_test_records,train_classes,test_classes,unsupported_queries,exact_text_overlap,template_overlap,equipment_overlap,row_overlap,representation_fit,failure_nodes_in_features
0,Random,9802,1500,2451,6,6,0,204,55,994,0,candidate records only,False


,split,method,hit@1,hit@5,hit@10,hit@20,hit@50,mrr@10,mrr@50
0,Equipment-disjoint,Graph context,0.9873,0.9960,0.9993,0.9993,1.0000,0.9915,0.9916
1,Equipment-disjoint,Hybrid RRF,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,Equipment-disjoint,Prior,0.2333,0.2333,0.2333,0.2333,0.2333,0.2333,0.2333
3,Equipment-disjoint,TF-IDF,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,Random,Graph context,0.9880,0.9967,0.9980,0.9987,0.9987,0.9917,0.9917
5,Random,Hybrid RRF,0.9980,1.0000,1.0000,1.0000,1.0000,0.9987,0.9987
6,Random,Prior,0.2587,0.2587,0.2587,0.2587,0.2587,0.2587,0.2587
7,Random,TF-IDF,0.9993,1.0000,1.0000,1.0000,1.0000,0.9997,0.9997
8,Template-disjoint,Graph context,0.9140,0.9900,0.9900,0.9947,1.0000,0.9337,0.9342
9,Template-disjoint,Hybrid RRF,0.9760,1.0000,1.0000,1.0000,1.0000,0.9879,0.9879


,split,candidate_records,query_records,full_test_records,train_classes,test_classes,unsupported_queries,exact_text_overlap,template_overlap,equipment_overlap,row_overlap,representation_fit,failure_nodes_in_features
0,Random,9802,1500,2451,6,6,0,204,55,994,0,candidate records only,False
1,Template-disjoint,9571,1500,2682,6,6,0,0,0,792,0,candidate records only,False
2,Equipment-disjoint,9822,1500,2431,6,6,0,199,52,0,0,candidate records only,False


## Findings passed forward

Group separation can remove exact overlaps without making text-derived failure labels independent. The next experiment therefore needs a structured target and explicit controls for identifier and template shortcuts.

In [5]:
findings = {
    "cohort_records": len(cohort), "failure_labels_derived_from_query_text": True,
    "random_template_overlap": int(random_audit["template_overlap"]),
    "random_exact_text_overlap": int(random_audit["exact_text_overlap"]),
    "random_equipment_overlap": int(random_audit["equipment_overlap"]),
    "splits": all_audits, "evidence_status": "diagnostic only, including group-disjoint results",
    "next_question": "Does retrieval still carry information about a structured asset target after identifier and template controls?"
}
write_json(out / "findings.json", findings)
display(findings)

{'cohort_records': 12253,
 'failure_labels_derived_from_query_text': True,
 'random_template_overlap': 55,
 'random_exact_text_overlap': 204,
 'random_equipment_overlap': 994,
 'splits': [{'split': 'Random',
   'candidate_records': 9802,
   'query_records': 1500,
   'full_test_records': 2451,
   'train_classes': 6,
   'test_classes': 6,
   'unsupported_queries': 0,
   'exact_text_overlap': 204,
   'template_overlap': 55,
   'equipment_overlap': 994,
   'row_overlap': 0,
   'representation_fit': 'candidate records only',
   'failure_nodes_in_features': False},
  {'split': 'Template-disjoint',
   'candidate_records': 9571,
   'query_records': 1500,
   'full_test_records': 2682,
   'train_classes': 6,
   'test_classes': 6,
   'unsupported_queries': 0,
   'exact_text_overlap': 0,
   'template_overlap': 0,
   'equipment_overlap': 792,
   'row_overlap': 0,
   'representation_fit': 'candidate records only',
   'failure_nodes_in_features': False},
  {'split': 'Equipment-disjoint',
   'candidat